<a href="https://colab.research.google.com/github/nnknishant/Pyspark_Project/blob/main/Fill_Null_Values_Pyspark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Example").getOrCreate()

In [2]:
# Create DataFrame

job_skills_data = [
    (1, 'Data Engineer', 'SQL'),
    (2, None, 'Python'),
    (3, None, 'AWS'),
    (4, None, 'Snowflake'),
    (5, None, 'Apache Spark'),
    (6, 'Web Developer', 'Java'),
    (7, None, 'HTML'),
    (8, None, 'CSS'),
    (9, 'Data Scientist', 'Python'),
    (10, None, 'Machine Learning'),
    (11, None, 'Deep Learning'),
    (12, None, 'Tableau')
]

In [3]:
job_skills_schema = "row_id int , job_role string , skills string"

In [4]:
job_skills_df = spark.createDataFrame(data = job_skills_data, schema=job_skills_schema)

In [5]:
job_skills_df.show()

+------+--------------+----------------+
|row_id|      job_role|          skills|
+------+--------------+----------------+
|     1| Data Engineer|             SQL|
|     2|          NULL|          Python|
|     3|          NULL|             AWS|
|     4|          NULL|       Snowflake|
|     5|          NULL|    Apache Spark|
|     6| Web Developer|            Java|
|     7|          NULL|            HTML|
|     8|          NULL|             CSS|
|     9|Data Scientist|          Python|
|    10|          NULL|Machine Learning|
|    11|          NULL|   Deep Learning|
|    12|          NULL|         Tableau|
+------+--------------+----------------+



In [13]:
# Import

from pyspark.sql.functions import col, when, first, sum
from pyspark.sql import window, Window



In [7]:
# Create flag column using when otherwise functions.

df1 = job_skills_df.withColumn("flag", when(col("job_role").isNotNull(), 1).otherwise(0))

df1.show()

+------+--------------+----------------+----+
|row_id|      job_role|          skills|flag|
+------+--------------+----------------+----+
|     1| Data Engineer|             SQL|   1|
|     2|          NULL|          Python|   0|
|     3|          NULL|             AWS|   0|
|     4|          NULL|       Snowflake|   0|
|     5|          NULL|    Apache Spark|   0|
|     6| Web Developer|            Java|   1|
|     7|          NULL|            HTML|   0|
|     8|          NULL|             CSS|   0|
|     9|Data Scientist|          Python|   1|
|    10|          NULL|Machine Learning|   0|
|    11|          NULL|   Deep Learning|   0|
|    12|          NULL|         Tableau|   0|
+------+--------------+----------------+----+



In [16]:
# Create group column

df2 = df1.withColumn("group", sum(col("flag")).over(Window.orderBy(col("row_id"))))

df2.show()

+------+--------------+----------------+----+-----+
|row_id|      job_role|          skills|flag|group|
+------+--------------+----------------+----+-----+
|     1| Data Engineer|             SQL|   1|    1|
|     2|          NULL|          Python|   0|    1|
|     3|          NULL|             AWS|   0|    1|
|     4|          NULL|       Snowflake|   0|    1|
|     5|          NULL|    Apache Spark|   0|    1|
|     6| Web Developer|            Java|   1|    2|
|     7|          NULL|            HTML|   0|    2|
|     8|          NULL|             CSS|   0|    2|
|     9|Data Scientist|          Python|   1|    3|
|    10|          NULL|Machine Learning|   0|    3|
|    11|          NULL|   Deep Learning|   0|    3|
|    12|          NULL|         Tableau|   0|    3|
+------+--------------+----------------+----+-----+



In [19]:
# Creating job_role column with first window function

df_answer = df2.withColumn("Job_Role", first(col("job_role")).over(Window.partitionBy(col("group")).orderBy(col("row_id"))))

df_answer.show()

+------+--------------+----------------+----+-----+
|row_id|      Job_Role|          skills|flag|group|
+------+--------------+----------------+----+-----+
|     1| Data Engineer|             SQL|   1|    1|
|     2| Data Engineer|          Python|   0|    1|
|     3| Data Engineer|             AWS|   0|    1|
|     4| Data Engineer|       Snowflake|   0|    1|
|     5| Data Engineer|    Apache Spark|   0|    1|
|     6| Web Developer|            Java|   1|    2|
|     7| Web Developer|            HTML|   0|    2|
|     8| Web Developer|             CSS|   0|    2|
|     9|Data Scientist|          Python|   1|    3|
|    10|Data Scientist|Machine Learning|   0|    3|
|    11|Data Scientist|   Deep Learning|   0|    3|
|    12|Data Scientist|         Tableau|   0|    3|
+------+--------------+----------------+----+-----+

